# EVT-CLIP++ Complete Training, Calibration, and Benchmark Notebook

This notebook builds the model side of the industrial anomaly-inspection system for five MVTec AD categories: **bottle, cable, capsule, metal_nut, and pill**.

It provides:

- Three category specialists: PatchCore, PaDiM, and FastFlow.
- A corrected VisionText DAEP–CMI network that always loads the real OpenCLIP backbone.
- CA-EVT-MR: category-aware Extreme Value Theory mask refinement and confidence calibration.
- Category compatibility checking and a VisionText fallback for images outside the trained categories.
- Pixel masks, true mask-derived bounding boxes and defect area, overlays, evaluation metrics, plots, and exportable artifacts.

**Research protocol note:** PatchCore/PaDiM/FastFlow learn only from normal images in their target category. The DAEP–CMI experiment below is cross-category supervised: it learns segmentation from source-category masks and is evaluated separately on a held-out category. Do not describe that experiment as strict zero-shot training. Unknown-category predictions are estimates and are explicitly labeled lower reliability.

Primary references: [Anomalib](https://github.com/open-edge-platform/anomalib), [Anomalib PyPI](https://pypi.org/project/anomalib/), and [OpenCLIP](https://github.com/mlfoundations/open_clip).



In [ ]:
#@title 1. Experiment configuration
from pathlib import Path

DATASET_DRIVE_ID = "1-sH1LPkXspIVAIdz6Tir8Nw5HsJMIEqi"
EXISTING_MODELS_DRIVE_ID = "1wgnwaHaXchGdv6na42UF_7-QAwsy95k6"

CATEGORIES = ["bottle", "cable", "capsule", "metal_nut", "pill"]
SOURCE_CATEGORIES = ["bottle", "cable", "metal_nut", "pill"]
HELD_OUT_CATEGORY = "capsule"
SPECIALISTS = ["patchcore", "padim", "fastflow"]

RETRAIN_ALL_SPECIALISTS = False #@param {type:"boolean"}
RUN_VISIONTEXT_TRAINING = True #@param {type:"boolean"}
FAST_DEMO_MODE = False #@param {type:"boolean"}
VISIONTEXT_EPOCHS = 30 #@param {type:"integer"}
SEED = 42

CONTENT_ROOT = Path("/content")
DATA_CACHE = CONTENT_ROOT / "evt_cache"
DATA_ROOT = CONTENT_ROOT / "mvtec"

print("Selected categories:", CATEGORIES)
print("Default behavior: reuse the existing 9 checkpoints and train the 6 missing metal_nut/pill checkpoints.")



In [ ]:
#@title 2. Mount Google Drive and require a GPU
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, sys
DRIVE_ROOT = Path("/content/drive/MyDrive/EVT_CLIP_PLUS_PLUS")
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
RESULT_ROOT = DRIVE_ROOT / "results"
CALIBRATION_ROOT = DRIVE_ROOT / "calibration"
for directory in (DATA_CACHE, DATA_ROOT, DRIVE_ROOT, CHECKPOINT_ROOT, RESULT_ROOT, CALIBRATION_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("No GPU is active. In Colab choose Runtime > Change runtime type > T4 GPU, then Run all again.")
print("GPU:", gpu.stdout.strip())
print("Persistent output:", DRIVE_ROOT)



In [ ]:
#@title 3. Install reproducible dependencies
%pip install -q "anomalib==2.6.0" "open-clip-torch==3.2.0" "lightning>=2.5,<2.7" gdown scipy scikit-learn pandas matplotlib seaborn opencv-python-headless

import anomalib, torch, open_clip, cv2, numpy as np
print("anomalib", anomalib.__version__)
print("torch", torch.__version__, "CUDA", torch.version.cuda, "available", torch.cuda.is_available())
print("open_clip", getattr(open_clip, "__version__", "installed"))
assert anomalib.__version__.startswith("2.6"), "This notebook was validated for anomalib 2.6.x"
assert torch.cuda.is_available(), "GPU is required for the full run"



In [ ]:
#@title 4. Download, extract, and validate MVTec AD
import gdown, tarfile, shutil, json, re

archive = DATA_CACHE / "mvtec_anomaly_detection.tar.xz"
if not archive.exists() or archive.stat().st_size < 1_000_000:
    print("Downloading the supplied MVTec archive...")
    gdown.download(id=DATASET_DRIVE_ID, output=str(archive), quiet=False, fuzzy=True)

sentinel = DATA_ROOT / ".extracted"
if not sentinel.exists():
    print("Extracting dataset (this takes several minutes)...")
    with tarfile.open(archive, "r:xz") as tar:
        tar.extractall(DATA_ROOT, filter="data")
    sentinel.touch()

def find_mvtec_root(base: Path) -> Path:
    matches = [p.parents[2] for p in base.rglob("bottle/train/good")]
    if not matches:
        raise FileNotFoundError("Could not find bottle/train/good after extraction")
    return matches[0]

MVTEC_ROOT = find_mvtec_root(DATA_ROOT)
IMAGE_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def count_images(path):
    return sum(p.suffix.lower() in IMAGE_EXT for p in Path(path).rglob("*"))

dataset_rows = []
for category in CATEGORIES:
    root = MVTEC_ROOT / category
    required = [root / "train/good", root / "test/good", root / "ground_truth"]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Invalid category {category}; missing {missing}")
    dataset_rows.append({
        "category": category,
        "train_good": count_images(root / "train/good"),
        "test_total": count_images(root / "test"),
        "ground_truth_masks": count_images(root / "ground_truth"),
    })

import pandas as pd
dataset_table = pd.DataFrame(dataset_rows)
display(dataset_table)
print("MVTec root:", MVTEC_ROOT)



In [ ]:
#@title 5. Import the existing nine trained checkpoints
weights_zip = DATA_CACHE / "vision_text_trained_models.zip"
existing_marker = CHECKPOINT_ROOT / ".existing_imported"
if not existing_marker.exists():
    if not weights_zip.exists() or weights_zip.stat().st_size < 1_000_000:
        gdown.download(id=EXISTING_MODELS_DRIVE_ID, output=str(weights_zip), quiet=False, fuzzy=True)
    unpack = DATA_CACHE / "existing_models"
    shutil.rmtree(unpack, ignore_errors=True)
    shutil.unpack_archive(str(weights_zip), str(unpack))
    aliases = {"patchcore": "patchcore", "padim": "padim", "fastflow": "fastflow"}
    imported = []
    for ckpt in unpack.rglob("*.ckpt"):
        low = str(ckpt).lower()
        model = next((v for k, v in aliases.items() if k in low), None)
        category = next((c for c in ["bottle", "cable", "capsule"] if c in low), None)
        if model and category:
            dst = CHECKPOINT_ROOT / model / f"{category}.ckpt"
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists():
                shutil.copy2(ckpt, dst)
            imported.append(str(dst))
    print("Imported", len(set(imported)), "existing specialist checkpoints")
    existing_marker.touch()

for model in SPECIALISTS:
    print(model, sorted(p.stem for p in (CHECKPOINT_ROOT / model).glob("*.ckpt")))



## Specialist training

Each specialist is trained independently per category using the category's normal training images. By default the notebook preserves bottle/cable/capsule from the supplied archive and trains only the missing `metal_nut` and `pill` combinations. Every completed checkpoint is copied to Drive immediately, so a disconnected Colab session can resume safely.



In [ ]:
#@title 6. Train/resume PatchCore, PaDiM, and FastFlow specialists
import inspect, time, gc
from anomalib.data import MVTecAD
from anomalib.engine import Engine
from anomalib.models import Patchcore, Padim, Fastflow

MODEL_FACTORY = {"patchcore": Patchcore, "padim": Padim, "fastflow": Fastflow}

def find_best_checkpoint(run_dir: Path) -> Path:
    candidates = list(run_dir.rglob("*.ckpt"))
    if not candidates:
        raise FileNotFoundError(f"Training ended without a checkpoint in {run_dir}")
    preferred = [p for p in candidates if "best" in p.name.lower()]
    return max(preferred or candidates, key=lambda p: p.stat().st_mtime)

def metrics_to_dict(result):
    if isinstance(result, list) and result:
        result = result[0]
    if not isinstance(result, dict):
        return {"raw_result": str(result)}
    out = {}
    for key, value in result.items():
        try:
            out[str(key)] = float(value.detach().cpu()) if hasattr(value, "detach") else float(value)
        except Exception:
            out[str(key)] = str(value)
    return out

def train_specialist(model_name: str, category: str):
    final_ckpt = CHECKPOINT_ROOT / model_name / f"{category}.ckpt"
    if final_ckpt.exists() and not RETRAIN_ALL_SPECIALISTS:
        return {"model": model_name, "category": category, "status": "reused", "checkpoint": str(final_ckpt)}

    run_dir = CONTENT_ROOT / "specialist_runs" / model_name / category
    shutil.rmtree(run_dir, ignore_errors=True)
    run_dir.mkdir(parents=True, exist_ok=True)
    datamodule = MVTecAD(
        root=MVTEC_ROOT,
        category=category,
        train_batch_size=1 if model_name == "patchcore" else 8,
        eval_batch_size=8,
        num_workers=2,
    )
    model = MODEL_FACTORY[model_name]()
    epochs = 1 if FAST_DEMO_MODE else (25 if model_name == "fastflow" else 1)
    engine = Engine(
        default_root_dir=run_dir,
        accelerator="gpu",
        devices=1,
        max_epochs=epochs,
        precision="16-mixed",
        deterministic=True,
    )
    started = time.perf_counter()
    engine.fit(model=model, datamodule=datamodule)
    test_result = engine.test(model=model, datamodule=datamodule)
    elapsed = time.perf_counter() - started
    source_ckpt = find_best_checkpoint(run_dir)
    final_ckpt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_ckpt, final_ckpt)
    row = {"model": model_name, "category": category, "status": "trained", "seconds": elapsed,
           "checkpoint": str(final_ckpt), **metrics_to_dict(test_result)}
    (RESULT_ROOT / "specialist_runs").mkdir(parents=True, exist_ok=True)
    with open(RESULT_ROOT / "specialist_runs" / f"{model_name}_{category}.json", "w") as f:
        json.dump(row, f, indent=2)
    del model, engine, datamodule
    gc.collect(); torch.cuda.empty_cache()
    return row

specialist_rows = []
for model_name in SPECIALISTS:
    for category in CATEGORIES:
        print(f"\n=== {model_name} / {category} ===")
        specialist_rows.append(train_specialist(model_name, category))

specialist_summary = pd.DataFrame(specialist_rows)
specialist_summary.to_csv(RESULT_ROOT / "specialist_training_summary.csv", index=False)
display(specialist_summary)



In [ ]:
#@title 6b. Evaluate all 15 specialist checkpoints on their matching categories
specialist_benchmarks = []
for model_name in SPECIALISTS:
    for category in CATEGORIES:
        ckpt = CHECKPOINT_ROOT / model_name / f"{category}.ckpt"
        if not ckpt.exists():
            specialist_benchmarks.append({"model": model_name, "category": category, "status": "missing"})
            continue
        print(f"Testing {model_name}/{category}")
        try:
            datamodule = MVTecAD(root=MVTEC_ROOT, category=category, train_batch_size=1, eval_batch_size=8, num_workers=2)
            loaded_model = MODEL_FACTORY[model_name].load_from_checkpoint(str(ckpt), map_location="cpu")
            test_engine = Engine(default_root_dir=CONTENT_ROOT / "specialist_test" / model_name / category,
                                 accelerator="gpu", devices=1, precision="16-mixed")
            started = time.perf_counter()
            tested = test_engine.test(model=loaded_model, datamodule=datamodule)
            specialist_benchmarks.append({"model": model_name, "category": category, "status": "tested",
                                          "test_seconds": time.perf_counter()-started, **metrics_to_dict(tested)})
            del loaded_model, test_engine, datamodule
            gc.collect(); torch.cuda.empty_cache()
        except Exception as exc:
            specialist_benchmarks.append({"model": model_name, "category": category, "status": "error", "error": repr(exc)})
            print("Evaluation error recorded:", repr(exc))

specialist_benchmark_table = pd.DataFrame(specialist_benchmarks)
specialist_benchmark_table.to_csv(RESULT_ROOT / "specialist_benchmarks.csv", index=False)
display(specialist_benchmark_table)



## Corrected VisionText DAEP–CMI network

The following implementation fails loudly if the real CLIP backbone cannot load. There is no random/synthetic feature fallback. OpenCLIP is frozen while DAEP, multi-level adapters, CMI, and the segmentation decoder are trained using source-category masks.



In [ ]:
#@title 7. Define the real OpenCLIP + DAEP + CMI model
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = torch.device("cuda")

class DynamicPrompt(nn.Module):
    def __init__(self, dim=768, tokens=4):
        super().__init__()
        self.context = nn.Parameter(torch.randn(2, tokens, dim) * 0.02)
        self.gate = nn.Sequential(nn.Linear(dim, dim // 4), nn.GELU(), nn.Linear(dim // 4, tokens), nn.Softmax(-1))
    def forward(self, image_global, text_pair):
        weights = self.gate(image_global).unsqueeze(-1)
        delta = (weights.unsqueeze(1) * self.context.unsqueeze(0)).sum(2)
        return F.normalize(text_pair + delta, dim=-1)

class CrossModalInteraction(nn.Module):
    def __init__(self, dim=256, heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, patches, text):
        attended, weights = self.attn(patches, text, text, need_weights=True)
        x = self.norm(patches + attended)
        return self.norm(x + self.ffn(x)), weights

class VisionTextDAEPCMI(nn.Module):
    def __init__(self, image_size=336, feature_dim=256):
        super().__init__()
        self.image_size = image_size
        try:
            self.clip, _, self.clip_preprocess = open_clip.create_model_and_transforms("ViT-L-14-336", pretrained="openai")
        except Exception as exc:
            raise RuntimeError("Real OpenCLIP ViT-L-14-336 failed to load; refusing a synthetic fallback") from exc
        self.tokenizer = open_clip.get_tokenizer("ViT-L-14-336")
        self.clip.eval().requires_grad_(False)
        clip_dim = int(self.clip.visual.output_dim)
        self.daep = DynamicPrompt(clip_dim)
        self.patch_adapter = nn.Sequential(nn.Linear(clip_dim, feature_dim), nn.LayerNorm(feature_dim), nn.GELU())
        self.text_adapter = nn.Sequential(nn.Linear(clip_dim, feature_dim), nn.LayerNorm(feature_dim))
        self.cmi = CrossModalInteraction(feature_dim)
        self.decoder = nn.Sequential(
            nn.Conv2d(feature_dim + 2, 128, 3, padding=1), nn.GroupNorm(8, 128), nn.GELU(),
            nn.Conv2d(128, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv2d(64, 1, 1),
        )

    def train(self, mode=True):
        super().train(mode)
        self.clip.eval()
        return self

    def encode_text_pair(self, categories):
        prompts = []
        for c in categories:
            name = c.replace("_", " ")
            prompts += [f"a photo of a clean normal {name}", f"a photo of a damaged defective {name}"]
        tokens = self.tokenizer(prompts).to(DEVICE)
        with torch.no_grad():
            emb = self.clip.encode_text(tokens, normalize=True)
        return emb.reshape(len(categories), 2, -1)

    def encode_visual_tokens(self, x):
        visual = self.clip.visual
        with torch.no_grad():
            x = visual.conv1(x)
            x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
            cls = visual.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], device=x.device, dtype=x.dtype)
            x = torch.cat([cls, x], dim=1)
            x = x + visual.positional_embedding.to(x.dtype)
            x = visual.patch_dropout(x)
            x = visual.ln_pre(x)
            # OpenCLIP 3.x ViT transformer consumes batch-first [N, L, D] tokens.
            x = visual.transformer(x)
            x = visual.ln_post(x)
            if visual.proj is not None:
                x = x @ visual.proj
        return x[:, 0], x[:, 1:]

    def forward(self, images, categories):
        global_feat, patch_feat = self.encode_visual_tokens(images)
        base_text = self.encode_text_pair(categories)
        prompt_text = self.daep(global_feat, base_text)
        p = self.patch_adapter(patch_feat)
        t = self.text_adapter(prompt_text)
        fused, attention = self.cmi(p, t)
        similarities = torch.einsum("bnd,bkd->bnk", F.normalize(p, dim=-1), F.normalize(t, dim=-1))
        side = int(round(p.shape[1] ** 0.5))
        if side * side != p.shape[1]:
            raise RuntimeError(f"Unexpected non-square CLIP patch grid: {p.shape[1]}")
        feature_map = fused.transpose(1, 2).reshape(images.shape[0], -1, side, side)
        sim_map = similarities.transpose(1, 2).reshape(images.shape[0], 2, side, side)
        logits = self.decoder(torch.cat([feature_map, sim_map], dim=1))
        logits = F.interpolate(logits, size=images.shape[-2:], mode="bilinear", align_corners=False)
        return {"logits": logits, "similarity": similarities, "attention": attention, "prompts": prompt_text}

visiontext = VisionTextDAEPCMI().to(DEVICE)
trainable = sum(p.numel() for p in visiontext.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in visiontext.parameters() if not p.requires_grad)
print(f"Trainable: {trainable:,}; frozen CLIP: {frozen:,}")
assert frozen > trainable, "CLIP backbone should be frozen"



In [ ]:
#@title 8. Build a leakage-safe cross-category segmentation dataset
import random
from torchvision import transforms

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
image_tf = transforms.Compose([transforms.Resize((336, 336)), transforms.ToTensor(), transforms.Normalize(CLIP_MEAN, CLIP_STD)])
mask_tf = transforms.Compose([transforms.Resize((336, 336), interpolation=transforms.InterpolationMode.NEAREST), transforms.ToTensor()])

def mask_for_test_image(image_path: Path):
    if image_path.parent.name == "good":
        return None
    candidate = image_path.parents[2] / "ground_truth" / image_path.parent.name / f"{image_path.stem}_mask.png"
    if not candidate.exists():
        hits = list((image_path.parents[2] / "ground_truth" / image_path.parent.name).glob(f"{image_path.stem}*"))
        return hits[0] if hits else None
    return candidate

class CrossCategoryDataset(Dataset):
    def __init__(self, categories, split="train", seed=42):
        rows = []
        for category in categories:
            for image_path in sorted((MVTEC_ROOT / category / "test").glob("*/*")):
                if image_path.suffix.lower() in IMAGE_EXT:
                    rows.append((image_path, mask_for_test_image(image_path), category))
        rng = random.Random(seed)
        rng.shuffle(rows)
        cut = int(0.8 * len(rows))
        self.rows = rows if split == "all" else rows[:cut] if split == "train" else rows[cut:]
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        image_path, mask_path, category = self.rows[idx]
        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") if mask_path else Image.new("L", image.size, 0)
        return image_tf(image), (mask_tf(mask) > 0.5).float(), category, str(image_path)

train_ds = CrossCategoryDataset(SOURCE_CATEGORIES, "train", SEED)
val_ds = CrossCategoryDataset(SOURCE_CATEGORIES, "val", SEED)
heldout_ds = CrossCategoryDataset([HELD_OUT_CATEGORY], "all", SEED)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=2)
heldout_loader = DataLoader(heldout_ds, batch_size=2, shuffle=False, num_workers=2)
print("source train", len(train_ds), "source validation", len(val_ds), "held-out", len(heldout_ds))



In [ ]:
#@title 9. Smoke-test, train, and resume VisionText
from torch.cuda.amp import autocast, GradScaler

def focal_dice_loss(logits, targets, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    pt = p * targets + (1 - p) * (1 - targets)
    focal = ((1 - pt) ** gamma * bce).mean()
    inter = (p * targets).sum((1,2,3))
    dice = 1 - ((2 * inter + 1) / (p.sum((1,2,3)) + targets.sum((1,2,3)) + 1)).mean()
    return focal + dice

VISIONTEXT_CKPT = CHECKPOINT_ROOT / "visiontext" / "daep_cmi_best.pt"
VISIONTEXT_CKPT.parent.mkdir(parents=True, exist_ok=True)
optimizer = torch.optim.AdamW((p for p in visiontext.parameters() if p.requires_grad), lr=2e-4, weight_decay=1e-4)
scaler = GradScaler()
start_epoch, best_val = 0, float("inf")
if VISIONTEXT_CKPT.exists():
    state = torch.load(VISIONTEXT_CKPT, map_location="cpu")
    visiontext.load_state_dict(state["model"], strict=False)
    start_epoch, best_val = state.get("epoch", -1) + 1, state.get("best_val", float("inf"))
    print("Resuming at epoch", start_epoch)

# Mandatory real-backbone smoke test.
batch = next(iter(train_loader))
with torch.no_grad(), autocast(dtype=torch.float16):
    smoke = visiontext(batch[0][:1].to(DEVICE), [batch[2][0]])
assert smoke["logits"].shape == (1, 1, 336, 336) and torch.isfinite(smoke["logits"]).all()
print("Real OpenCLIP forward smoke test passed")

def validation_loss(loader):
    visiontext.eval(); losses = []
    with torch.no_grad():
        for images, masks, cats, _ in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            with autocast(dtype=torch.float16):
                losses.append(float(focal_dice_loss(visiontext(images, list(cats))["logits"], masks)))
    return float(np.mean(losses))

if RUN_VISIONTEXT_TRAINING:
    epochs = min(VISIONTEXT_EPOCHS, 1) if FAST_DEMO_MODE else VISIONTEXT_EPOCHS
    history = []
    for epoch in range(start_epoch, epochs):
        visiontext.train(); optimizer.zero_grad(set_to_none=True); running = []
        for step, (images, masks, cats, _) in enumerate(train_loader):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            with autocast(dtype=torch.float16):
                loss = focal_dice_loss(visiontext(images, list(cats))["logits"], masks) / 8
            scaler.scale(loss).backward()
            if (step + 1) % 8 == 0 or step + 1 == len(train_loader):
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            running.append(float(loss) * 8)
        val_loss = validation_loss(val_loader)
        row = {"epoch": epoch, "train_loss": float(np.mean(running)), "val_loss": val_loss}
        history.append(row); print(row)
        if val_loss < best_val:
            best_val = val_loss
            trainable_state = {k: v.detach().cpu() for k, v in visiontext.state_dict().items() if not k.startswith("clip.")}
            torch.save({"model": trainable_state, "epoch": epoch, "best_val": best_val,
                        "protocol": "cross-category supervised", "sources": SOURCE_CATEGORIES,
                        "held_out": HELD_OUT_CATEGORY}, VISIONTEXT_CKPT)
        pd.DataFrame(history).to_csv(RESULT_ROOT / "visiontext_training_history.csv", index=False)

print("VisionText checkpoint:", VISIONTEXT_CKPT)



## Evaluation and CA-EVT-MR

CA-EVT-MR fits a Generalized Pareto Distribution to the upper tail of normal pixel scores for each category. Tail probabilities calibrate the mask threshold. Morphology and connected-component filtering remove isolated noise. Bounding boxes, defect surface area, and severity are computed from the resulting binary mask—never from the entire image.



In [ ]:
#@title 10. Metrics, AUPRO, and CA-EVT-MR implementation
from scipy.stats import genpareto
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve, confusion_matrix
from scipy import ndimage

def f1_max(y, s):
    p, r, t = precision_recall_curve(y, s)
    f = 2 * p * r / np.maximum(p + r, 1e-12)
    i = int(np.nanargmax(f))
    return float(f[i]), float(t[min(i, len(t)-1)]) if len(t) else 0.5

def overlap_metrics(gt, pred):
    gt, pred = gt.astype(bool), pred.astype(bool)
    inter = np.logical_and(gt, pred).sum(); union = np.logical_or(gt, pred).sum()
    return {"iou": float(inter / max(union, 1)), "dice": float(2 * inter / max(gt.sum()+pred.sum(), 1)),
            "pixel_accuracy": float((gt == pred).mean())}

def au_pro(masks, maps, max_fpr=0.30, steps=100):
    normal = ~masks.astype(bool)
    pros, fprs = [], []
    for threshold in np.linspace(float(maps.max()), float(maps.min()), steps):
        pred = maps >= threshold
        fpr = np.logical_and(pred, normal).sum() / max(normal.sum(), 1)
        region_scores = []
        for gt, pr in zip(masks, pred):
            labels, n = ndimage.label(gt)
            for label in range(1, n + 1):
                region = labels == label
                region_scores.append(np.logical_and(pr, region).sum() / region.sum())
        fprs.append(fpr); pros.append(np.mean(region_scores) if region_scores else 0)
    order = np.argsort(fprs); x, y = np.array(fprs)[order], np.array(pros)[order]
    keep = x <= max_fpr
    return float(np.trapz(y[keep], x[keep]) / max_fpr) if keep.sum() > 1 else float("nan")

class CAEVMR:
    def __init__(self, quantile=0.95, alpha=0.01, min_component=16):
        self.quantile, self.alpha, self.min_component = quantile, alpha, min_component
        self.params = {}
    def fit(self, category, normal_maps):
        values = np.asarray(normal_maps, np.float32).ravel()
        u = float(np.quantile(values, self.quantile))
        excess = values[values > u] - u
        if len(excess) < 50:
            raise ValueError(f"Insufficient normal-tail samples for {category}")
        shape, loc, scale = genpareto.fit(excess, floc=0)
        self.params[category] = {"u": u, "shape": float(shape), "scale": float(scale),
                                 "quantile": self.quantile, "alpha": self.alpha}
        return self.params[category]
    def refine(self, category, score_map):
        p = self.params[category]; excess = np.maximum(score_map - p["u"], 0)
        tail_survival = genpareto.sf(excess, p["shape"], loc=0, scale=max(p["scale"], 1e-8))
        calibrated = np.where(score_map > p["u"], 1 - tail_survival, 0)
        mask = calibrated >= (1 - self.alpha)
        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
        labels, n, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
        clean = np.zeros_like(mask)
        for i in range(1, n):
            if stats[i, cv2.CC_STAT_AREA] >= self.min_component:
                clean[labels == i] = 1
        return calibrated.astype(np.float32), clean
    def save(self, path):
        with open(path, "w") as f: json.dump(self.params, f, indent=2)
    def load(self, path):
        with open(path) as f: self.params = json.load(f)
        return self

def defect_geometry(mask):
    mask = mask.astype(np.uint8)
    ys, xs = np.where(mask > 0)
    if not len(xs):
        return {"bbox": None, "area_pixels": 0, "area_percent_image": 0.0, "centroid": None, "location": "none"}
    x1, x2, y1, y2 = int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())
    cx, cy = float(xs.mean()), float(ys.mean()); h, w = mask.shape
    horizontal = "left" if cx < w/3 else "right" if cx > 2*w/3 else "center"
    vertical = "top" if cy < h/3 else "bottom" if cy > 2*h/3 else "middle"
    return {"bbox": [x1, y1, x2-x1+1, y2-y1+1], "area_pixels": int(mask.sum()),
            "area_percent_image": float(100*mask.mean()), "centroid": [cx, cy], "location": f"{vertical} {horizontal}"}

def severity(area_pct, confidence):
    risk = area_pct * max(confidence, 0.25)
    return "Critical" if risk >= 15 else "High" if risk >= 5 else "Medium" if risk >= 1 else "Low"



In [ ]:
#@title 11. VisionText score maps and held-out benchmark
def load_best_visiontext():
    if not VISIONTEXT_CKPT.exists():
        raise FileNotFoundError("Train VisionText first")
    state = torch.load(VISIONTEXT_CKPT, map_location="cpu")
    visiontext.load_state_dict(state["model"], strict=False)
    visiontext.to(DEVICE).eval()

def predict_visiontext_tensor(images, categories):
    with torch.no_grad(), autocast(dtype=torch.float16):
        out = visiontext(images.to(DEVICE), list(categories))
        maps = torch.sigmoid(out["logits"]).float().cpu().numpy()[:, 0]
    scores = np.array([np.mean(np.partition(m.ravel(), -max(1, m.size//100))[-max(1, m.size//100):]) for m in maps])
    return maps, scores, out

def benchmark_visiontext(loader, label):
    all_maps, all_masks, all_scores, image_labels, paths = [], [], [], [], []
    times = []
    for images, masks, cats, batch_paths in loader:
        start = time.perf_counter(); maps, scores, _ = predict_visiontext_tensor(images, cats)
        times.append(1000*(time.perf_counter()-start)/len(images))
        all_maps.extend(maps); all_masks.extend(masks.numpy()[:,0]); all_scores.extend(scores)
        image_labels.extend((masks.numpy().reshape(len(masks), -1).max(1) > 0).astype(int)); paths.extend(batch_paths)
    maps, masks, scores, labels = map(np.asarray, (all_maps, all_masks, all_scores, image_labels))
    pix_gt, pix_score = masks.ravel().astype(int), maps.ravel()
    f1, threshold = f1_max(pix_gt, pix_score); pred = maps >= threshold
    row = {"method": "VisionText-DAEP-CMI", "split": label,
           "image_auroc": roc_auc_score(labels, scores), "image_ap": average_precision_score(labels, scores),
           "pixel_auroc": roc_auc_score(pix_gt, pix_score), "pixel_ap": average_precision_score(pix_gt, pix_score),
           "f1_max": f1, "threshold": threshold, "aupro_0.30": au_pro(masks > .5, maps),
           "inference_ms_mean": float(np.mean(times)), **overlap_metrics(masks > .5, pred)}
    return row, maps, masks, scores, labels, paths

if VISIONTEXT_CKPT.exists():
    load_best_visiontext()
    heldout_result = benchmark_visiontext(heldout_loader, f"held-out:{HELD_OUT_CATEGORY}")
    display(pd.DataFrame([heldout_result[0]]))
    pd.DataFrame([heldout_result[0]]).to_csv(RESULT_ROOT / "visiontext_heldout_metrics.csv", index=False)
else:
    print("VisionText checkpoint not present; run training cell before evaluation.")



In [ ]:
#@title 12. Fit category-specific EVT calibrators on normal images
def normal_loader_for(category, limit=None):
    files = sorted((MVTEC_ROOT / category / "train/good").glob("*"))
    if limit: files = files[:limit]
    for path in files:
        image = Image.open(path).convert("RGB")
        yield image_tf(image).unsqueeze(0), [category]

evt = CAEVMR(quantile=0.95, alpha=0.01, min_component=16)
EVT_PATH = CALIBRATION_ROOT / "ca_evt_mr.json"
if VISIONTEXT_CKPT.exists():
    load_best_visiontext()
    for category in CATEGORIES:
        normal_maps = []
        for image, cats in normal_loader_for(category, 20 if FAST_DEMO_MODE else None):
            maps, _, _ = predict_visiontext_tensor(image, cats)
            normal_maps.append(maps[0])
        print(category, evt.fit(category, normal_maps))
    evt.save(EVT_PATH)
    print("Saved", EVT_PATH)



In [ ]:
#@title 13. Generate benchmark plots and qualitative outputs
import matplotlib.pyplot as plt
import seaborn as sns

PLOT_ROOT = RESULT_ROOT / "plots"; PLOT_ROOT.mkdir(parents=True, exist_ok=True)
SAMPLE_ROOT = RESULT_ROOT / "samples"; SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)

def save_visual(image_path, score_map, mask, destination):
    rgb = np.asarray(Image.open(image_path).convert("RGB").resize((score_map.shape[1], score_map.shape[0])))
    heat = cv2.applyColorMap(np.uint8(np.clip(score_map,0,1)*255), cv2.COLORMAP_JET)[:,:,::-1]
    overlay = np.uint8(0.58*rgb + 0.42*heat)
    geom = defect_geometry(mask)
    if geom["bbox"]:
        x,y,w,h = geom["bbox"]; cv2.rectangle(overlay, (x,y), (x+w-1,y+h-1), (255,255,255), 2)
    fig, axes = plt.subplots(1,4,figsize=(14,4))
    for ax, arr, title, cmap in zip(axes, [rgb, heat, mask, overlay], ["Original","Heatmap","CA-EVT-MR Mask","Overlay"], [None,None,"gray",None]):
        ax.imshow(arr, cmap=cmap); ax.set_title(title); ax.axis("off")
    fig.tight_layout(); fig.savefig(destination, dpi=180, bbox_inches="tight"); plt.close(fig)
    return geom

if VISIONTEXT_CKPT.exists():
    row, maps, masks, scores, labels, paths = heldout_result
    if EVT_PATH.exists(): evt.load(EVT_PATH)
    predictions = []
    for i in range(min(12, len(paths))):
        calibrated, refined = evt.refine(HELD_OUT_CATEGORY, maps[i])
        geom = save_visual(paths[i], calibrated, refined, SAMPLE_ROOT / f"capsule_{i:02d}.png")
        predictions.append({"image": paths[i], "score": float(scores[i]), "label": int(labels[i]), **geom})
    pd.DataFrame(predictions).to_csv(RESULT_ROOT / "sample_predictions.csv", index=False)

    fpr, tpr, _ = roc_curve(labels, scores); precision, recall, _ = precision_recall_curve(labels, scores)
    fig, axes = plt.subplots(1,2,figsize=(10,4))
    axes[0].plot(fpr,tpr); axes[0].plot([0,1],[0,1],'--'); axes[0].set(title="Image ROC",xlabel="FPR",ylabel="TPR")
    axes[1].plot(recall,precision); axes[1].set(title="Image Precision–Recall",xlabel="Recall",ylabel="Precision")
    fig.tight_layout(); fig.savefig(PLOT_ROOT / "heldout_curves.png",dpi=180); plt.show()



## Unknown/untrained image inspection

Category compatibility is measured with CLIP before inference. If similarity is weak, the output is marked **unseen-category estimate (lower reliability)** and routed to the category-agnostic VisionText path. This makes the UI return a result for arbitrary images without pretending that an unsupported product has specialist-level reliability.



In [ ]:
#@title 14. Upload and inspect any image
from google.colab import files

def category_compatibility(pil_image):
    prompts = [f"a photo of an industrial {c.replace('_',' ')} product" for c in CATEGORIES]
    tokens = visiontext.tokenizer(prompts).to(DEVICE)
    with torch.no_grad():
        text = visiontext.clip.encode_text(tokens, normalize=True)
        image = image_tf(pil_image).unsqueeze(0).to(DEVICE)
        img = visiontext.clip.encode_image(image, normalize=True)
        prob = (100 * img @ text.T).softmax(-1)[0].float().cpu().numpy()
    idx = int(prob.argmax())
    return CATEGORIES[idx], float(prob[idx]), dict(zip(CATEGORIES, map(float, prob)))

def inspect_unknown(path, selected_category=None):
    if not VISIONTEXT_CKPT.exists() or not EVT_PATH.exists():
        raise RuntimeError("Train/load VisionText and fit CA-EVT-MR first")
    load_best_visiontext(); evt.load(EVT_PATH)
    pil = Image.open(path).convert("RGB")
    candidate, compatibility, all_compat = category_compatibility(pil)
    category = selected_category or candidate
    mismatch = selected_category is not None and selected_category != candidate
    supported = compatibility >= 0.45 and not mismatch
    start = time.perf_counter()
    maps, scores, out = predict_visiontext_tensor(image_tf(pil).unsqueeze(0), [category])
    calibrated, mask = evt.refine(category, maps[0])
    elapsed_ms = 1000 * (time.perf_counter() - start)
    score = float(scores[0]); geom = defect_geometry(mask)
    confidence = float(np.clip((score - evt.params[category]["u"]) / max(1-evt.params[category]["u"],1e-6), 0, 1))
    prediction = "Defective Product" if mask.any() else "Good Product"
    reliability = "supported-category estimate" if supported else "unseen/mismatched-category estimate — lower reliability"
    result = {"prediction": prediction, "confidence": confidence, "anomaly_score": score,
              "category_used": category, "category_candidate": candidate, "category_compatibility": compatibility,
              "reliability": reliability, "severity": severity(geom["area_percent_image"], confidence),
              "processing_ms": elapsed_ms, "category_scores": all_compat, **geom}
    out_path = RESULT_ROOT / f"inspection_{Path(path).stem}.png"
    save_visual(path, calibrated, mask, out_path)
    with open(RESULT_ROOT / f"inspection_{Path(path).stem}.json", "w") as f: json.dump(result, f, indent=2)
    display(pd.DataFrame([{k:v for k,v in result.items() if k != "category_scores"}]).T.rename(columns={0:"value"}))
    display(Image.open(out_path))
    if mismatch:
        print(f"WARNING: selected {selected_category}, but CLIP sees {candidate}. Use the matching category when possible.")
    return result

uploaded = files.upload()
for name, payload in uploaded.items():
    local = CONTENT_ROOT / name
    local.write_bytes(payload)
    inspect_unknown(local)



In [ ]:
#@title 15. Consolidate artifacts, benchmark table, and export
import platform, zipfile

metric_files = list(RESULT_ROOT.rglob("*metrics*.csv")) + list((RESULT_ROOT / "specialist_runs").glob("*.json"))
manifest = {
    "project": "EVT-CLIP++",
    "categories": CATEGORIES,
    "specialists": SPECIALISTS,
    "visiontext_protocol": "cross-category supervised; capsule held out",
    "innovation": "CA-EVT-MR category-aware EVT mask refinement",
    "unknown_policy": "CLIP compatibility check + VisionText lower-reliability fallback",
    "anomalib": anomalib.__version__, "torch": torch.__version__, "gpu": gpu.stdout.strip(),
    "dataset_drive_id": DATASET_DRIVE_ID,
    "checkpoint_inventory": [str(p.relative_to(DRIVE_ROOT)) for p in CHECKPOINT_ROOT.rglob("*") if p.is_file()],
    "metric_files": [str(p.relative_to(DRIVE_ROOT)) for p in metric_files if p.exists()],
}
with open(DRIVE_ROOT / "model_manifest.json", "w") as f: json.dump(manifest, f, indent=2)

export_zip = DRIVE_ROOT / "EVT_CLIP_PLUS_PLUS_ARTIFACTS.zip"
with zipfile.ZipFile(export_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (RESULT_ROOT, CALIBRATION_ROOT):
        for path in folder.rglob("*"):
            if path.is_file(): z.write(path, path.relative_to(DRIVE_ROOT))
    z.write(DRIVE_ROOT / "model_manifest.json", "model_manifest.json")
print("Completed. Persistent artifacts:", DRIVE_ROOT)
print("Portable report bundle:", export_zip)
